This is the code for downloading the dataset for Prjoect 1

In [1]:
# --- Imports & Setup ---
import wrds
import pandas as pd
import os
import numpy as np

# --- Connect to WRDS ---
wrds_db = wrds.Connection()

# --- Download CRSP monthly returns ---
crsp = wrds_db.raw_sql("""
    SELECT permno, date, ret, shrout, prc, altprc, vol, retx
    FROM crsp.msf
    WHERE date >= '1990-01-01'
""")

crsp['date'] = pd.to_datetime(crsp['date'])
crsp['yyyymm'] = crsp['date'].dt.year * 100 + crsp['date'].dt.month
crsp['me'] = crsp['prc'].abs() * crsp['shrout']
crsp['logme'] = np.log(crsp['me'])

# --- Load predictive characteristics (from jkpfactors.com) ---
chars = pd.read_csv("signed_predictors_dl_wide.csv")

# --- Merge datasets ---
merged = pd.merge(chars, crsp, on=['permno', 'yyyymm'], how='inner')
merged = merged.fillna(0)
merged.columns = merged.columns.str.lower()


# --- Standardize features cross-sectionally by month ---
exclude_cols = ['permno', 'yyyymm', 'date', 'ret']
feature_cols = [col for col in merged.columns if col not in exclude_cols]
merged[feature_cols] = merged.groupby('yyyymm')[feature_cols].transform(
    lambda x: (x - x.mean()) / x.std()
)
merged = merged.fillna(0)

# --- Save yearly .parquet files ---
os.makedirs("dataset_yearly_parquet", exist_ok=True)
merged['year'] = merged['yyyymm'] // 100

for year in sorted(merged['year'].unique()):
    chunk = merged[merged['year'] == year]
    filename = f"dataset_yearly_parquet/prepared_{year}.parquet"
    chunk.to_parquet(filename, index=False)
    print(f"Saved {filename} with shape {chunk.shape}")


WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


/Users/marcusdehaan/miniconda3/envs/myenv/lib/python3.12/site-packages/pandas/core/arrays/masked.py:672: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs2, **kwargs)
/var/folders/1n/j1f1sg_s3pqg792t3g815f580000gn/T/ipykernel_1901/2272809554.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged['year'] = merged['yyyymm'] // 100


Saved dataset_yearly_parquet/prepared_1990.parquet with shape (82997, 221)
Saved dataset_yearly_parquet/prepared_1991.parquet with shape (82127, 221)
Saved dataset_yearly_parquet/prepared_1992.parquet with shape (84423, 221)
Saved dataset_yearly_parquet/prepared_1993.parquet with shape (89570, 221)
Saved dataset_yearly_parquet/prepared_1994.parquet with shape (98478, 221)
Saved dataset_yearly_parquet/prepared_1995.parquet with shape (101307, 221)
Saved dataset_yearly_parquet/prepared_1996.parquet with shape (106992, 221)
Saved dataset_yearly_parquet/prepared_1997.parquet with shape (110908, 221)
Saved dataset_yearly_parquet/prepared_1998.parquet with shape (109858, 221)
Saved dataset_yearly_parquet/prepared_1999.parquet with shape (104677, 221)
Saved dataset_yearly_parquet/prepared_2000.parquet with shape (102573, 221)
Saved dataset_yearly_parquet/prepared_2001.parquet with shape (95742, 221)
Saved dataset_yearly_parquet/prepared_2002.parquet with shape (89277, 221)
Saved dataset_yearl

# **If you have already downloaded the yearly parquet files from GitHub, ONLY RUN THIS BLOCK
These are some diagnostic tests to ensure you data is working properly.

In [3]:
# --- Imports (if needed) ---
import pandas as pd
import glob
from sklearn.linear_model import LinearRegression
import numpy as np

# --- Load full dataset ---
all_files = glob.glob("dataset_yearly_parquet/*.parquet")
df = pd.concat([pd.read_parquet(f) for f in all_files])
print("Full dataset shape:", df.shape)

# --- Define features ---
exclude_cols = ['permno', 'yyyymm', 'date', 'ret']
feature_cols = [col for col in df.columns if col not in exclude_cols]

# --- Quick check: standardization across time ---
sample_cols = feature_cols[:5]
print(df.groupby('yyyymm')[sample_cols].agg(['mean', 'std']).head())

# --- Replicate 3-factor OLS model ---
factors = ['logme', 'bm', 'mom12m']
assert all(f in df.columns for f in factors), "Missing one of the 3 benchmark factors"

X = df[factors]
y = df['ret']
mask = X.notnull().all(axis=1) & y.notnull()
X = X[mask]
y = y[mask]

model = LinearRegression()
model.fit(X, y)
r2 = model.score(X, y)

print(f"Benchmark OLS (3-factor) R²: {r2 * 100:.4f}%")

# --- Duplication check ---
dupes = df.duplicated(subset=['permno', 'yyyymm'])
print(f"Duplicate rows by permno + yyyymm: {dupes.sum()}")

Full dataset shape: (3185565, 221)
                  am                aop      abnormalaccruals       \
                mean  std          mean  std             mean  std   
yyyymm                                                               
199001 -2.861670e-18  1.0 -1.681808e-18  1.0    -5.431307e-19  1.0   
199002 -8.544684e-19  1.0 -1.448065e-18  1.0     1.313873e-18  1.0   
199003 -1.382319e-17  1.0 -7.694056e-18  1.0     1.022742e-17  1.0   
199004 -8.162890e-19  1.0  2.814118e-18  1.0     1.606355e-18  1.0   
199005  7.335374e-19  1.0  3.465952e-18  1.0     2.340928e-18  1.0   

            accruals         accrualsbm       
                mean  std          mean  std  
yyyymm                                        
199001  1.003993e-17  1.0 -4.217250e-18  1.0  
199002 -5.776236e-18  1.0  5.640042e-18  1.0  
199003  1.574495e-17  1.0  9.051948e-18  1.0  
199004 -2.565373e-18  1.0  6.073780e-18  1.0  
199005  1.760090e-17  1.0 -8.029463e-18  1.0  
Benchmark OLS (3-factor) R²:

In [ ]:
#This prints all of the variable names in the dataset
import pprint
pprint.pprint(sorted(df.columns.tolist()))

['abnormalaccruals',
 'accruals',
 'accrualsbm',
 'activism1',
 'activism2',
 'adexp',
 'ageipo',
 'altprc',
 'am',
 'analystrevision',
 'analystvalue',
 'announcementreturn',
 'aop',
 'assetgrowth',
 'beta',
 'betafp',
 'betaliquidityps',
 'betatailrisk',
 'betavix',
 'bidaskspread',
 'bm',
 'bmdec',
 'bookleverage',
 'bpebm',
 'brandinvest',
 'cash',
 'cashprod',
 'cboperprof',
 'cf',
 'cfp',
 'changeinrecommendation',
 'chassetturnover',
 'cheq',
 'chforecastaccrual',
 'chinv',
 'chinvia',
 'chnanalyst',
 'chnncoa',
 'chnwc',
 'chtax',
 'citationsrd',
 'compequiss',
 'compositedebtissuance',
 'consrecomm',
 'convdebt',
 'coskewacx',
 'coskewness',
 'cpvolspread',
 'credratdg',
 'customermomentum',
 'date',
 'dcpvolspread',
 'debtissuance',
 'delbreadth',
 'delcoa',
 'delcol',
 'deldrc',
 'delequ',
 'delfinl',
 'dellti',
 'delnetfin',
 'divinit',
 'divomit',
 'divseason',
 'divyieldst',
 'dnoa',
 'dolvol',
 'downrecomm',
 'dvolcall',
 'dvolput',
 'earningsconsistency',
 'earningsfore